# Module 2: Convolutional Building Blocks

Welcome to the heart of how neural networks see! Convolutions are the fundamental operation behind every image model you've heard of — and they're the backbone of the U-Net architecture we'll build for diffusion.

In this module, you'll implement these building blocks from scratch:

- **2D convolution** with multiple input/output channels
- **Padding, stride, and dilation** controls
- **Transposed convolutions** for upsampling
- **Depthwise separable convolutions** for efficiency
- **GroupNorm** (the normalization layer diffusion models actually use)
- **Residual connections** with 1x1 projection shortcuts

Each section pairs a worked example with an exercise. At the end, you'll combine everything into a small ConvNet that classifies CIFAR-10 — a simplified version of the U-Net encoder you'll build in Module 4.

Let's get started.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
from typing import Optional, Tuple, Union
import time

torch.manual_seed(42)
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

---
## 2.1 — What Is a Convolution?

A convolution slides a small filter (kernel) across an input and computes a weighted sum at each position. Think of it like a flashlight scanning across an image — at each position, the filter "looks at" a small patch and produces a single number summarizing what it sees.

### Why convolutions work so well for images

Three properties make convolutions the right tool for spatial data:

### Weight sharing

The same kernel weights are reused at every spatial position. A 3x3 kernel has just 9 parameters, regardless of whether the input is 32x32 or 1024x1024. This is dramatically more efficient than a fully-connected layer.

### Translation equivariance

If the input shifts right by 2 pixels, the output shifts right by 2 pixels. The network doesn't need to "re-learn" a pattern at every possible location.

### Local receptive field

Each output element depends only on a small neighborhood of the input. The network builds up global understanding by stacking multiple layers of local operations.

### The convolution equation (2D, single channel)

$$y[i, j] = \sum_{m=0}^{K_h-1} \sum_{n=0}^{K_w-1} x[i+m,\; j+n] \cdot w[m, n]$$

In plain English: place the kernel at position $(i, j)$, multiply each kernel weight by the corresponding input pixel, and sum them up. That sum becomes the output at position $(i, j)$.

For example, a `(1, 8, 8)` input with a `(3, 3)` kernel produces a `(1, 6, 6)` output — you lose 2 pixels on each side because the kernel can't extend past the border.

> **Note:** In deep learning, "convolution" actually means **cross-correlation** (no kernel flip). PyTorch's `nn.Conv2d` implements cross-correlation.

### Worked example: 2D convolution with unfolding

Let's implement a basic 2D convolution using `unfold`, which extracts all sliding windows at once. This avoids explicit loops and shows you exactly what the convolution is computing.

In [ ]:
def conv2d_naive(
    x: torch.Tensor,
    kernel: torch.Tensor,
) -> torch.Tensor:
    """2D convolution (correlation) for a single-channel input using unfold.
    
    Args:
        x: Input of shape (H, W)
        kernel: Filter of shape (K_h, K_w)
    
    Returns:
        Output of shape (H - K_h + 1, W - K_w + 1)
    """
    H, W = x.shape
    K_h, K_w = kernel.shape
    
    # Use unfold to extract all sliding windows
    # First unfold along height, then width
    patches = x.unfold(0, K_h, 1).unfold(1, K_w, 1)  # (H_out, W_out, K_h, K_w)
    return (patches * kernel).sum(dim=(-2, -1))  # (H_out, W_out)


# Test against F.conv2d
torch.manual_seed(42)
x_2d = torch.randn(6, 6)  # (H, W)
k_2d = torch.randn(3, 3)  # (K_h, K_w)

out_ours = conv2d_naive(x_2d, k_2d)  # (4, 4)
out_pt = F.conv2d(
    x_2d.view(1, 1, 6, 6),  # (B, C, H, W)
    k_2d.view(1, 1, 3, 3)   # (C_out, C_in, K_h, K_w)
).squeeze()                  # (4, 4)

print(f"Our output shape: {out_ours.shape}")
print(f"Max error: {(out_ours - out_pt).abs().max():.2e}")
assert torch.allclose(out_ours, out_pt, atol=1e-5)
print("2D convolution matches F.conv2d ✓")

### Multi-channel convolution

With multiple input channels and output channels, the kernel becomes a 4D tensor: `(C_out, C_in, K_h, K_w)`.

Each output channel has its own set of `C_in` filters. For each output channel:
- Convolve each input channel with the corresponding 2D kernel slice
- Sum the results across input channels
- Add the bias (if any)

$$y[c_{\text{out}}, i, j] = b[c_{\text{out}}] + \sum_{c_{\text{in}}=0}^{C_{\text{in}}-1} \sum_{m,n} x[c_{\text{in}}, i+m, j+n] \cdot w[c_{\text{out}}, c_{\text{in}}, m, n]$$

**Concrete shapes:** A `(3, 32, 32)` RGB input with 16 filters of size `(3, 3)` produces a `(16, 30, 30)` output. The weight tensor has shape `(16, 3, 3, 3)` — one `(3, 3, 3)` filter per output channel.

### Exercise 2.1: Multi-channel 2D convolution from scratch

**Implement `conv2d_multichannel`** that handles `(B, C_in, H, W)` inputs with `(C_out, C_in, K_h, K_w)` kernels.

- Use `F.unfold` to extract sliding windows, then matrix-multiply with the reshaped kernel
- Handle an optional bias of shape `(C_out,)`
- The output shape should be `(B, C_out, H_out, W_out)`

In [ ]:
# YOUR CODE HERE — Exercise 2.1

def conv2d_multichannel(
    x: torch.Tensor,
    weight: torch.Tensor,
    bias: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Multi-channel 2D convolution from scratch.
    
    Args:
        x: Input tensor of shape (B, C_in, H, W)
        weight: Kernel tensor of shape (C_out, C_in, K_h, K_w)
        bias: Optional bias of shape (C_out,)
    
    Returns:
        Output tensor of shape (B, C_out, H_out, W_out)
    """
    B, C_in, H, W = x.shape
    C_out, C_in_k, K_h, K_w = weight.shape
    H_out = H - K_h + 1
    W_out = W - K_w + 1
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
torch.manual_seed(42)
x_test = torch.randn(2, 3, 8, 8)   # (B, C_in, H, W)
w_test = torch.randn(4, 3, 3, 3)   # (C_out, C_in, K_h, K_w)
b_test = torch.randn(4)             # (C_out,)

out_custom = conv2d_multichannel(x_test, w_test, b_test)
out_ref = F.conv2d(x_test, w_test, b_test)
assert out_custom is not None, "conv2d_multichannel returned None — did you forget to return?"
assert out_custom.shape == (2, 4, 6, 6), f"Expected shape (2, 4, 6, 6) but got {out_custom.shape}"
assert torch.allclose(out_custom, out_ref, atol=1e-4), f"Max error: {(out_custom - out_ref).abs().max():.2e} — check your unfold + matmul logic"
print(f"Output shape: {out_custom.shape}")  # (2, 4, 6, 6)
print("Multi-channel conv matches F.conv2d ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def conv2d_multichannel(
    x: torch.Tensor,
    weight: torch.Tensor,
    bias: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Multi-channel 2D convolution from scratch using unfold.
    
    Args:
        x: Input tensor of shape (B, C_in, H, W)
        weight: Kernel tensor of shape (C_out, C_in, K_h, K_w)
        bias: Optional bias of shape (C_out,)
    
    Returns:
        Output tensor of shape (B, C_out, H_out, W_out)
    """
    B, C_in, H, W = x.shape  # (B, C_in, H, W)
    C_out, C_in_k, K_h, K_w = weight.shape  # (C_out, C_in, K_h, K_w)
    assert C_in == C_in_k, f"Channel mismatch: input has {C_in}, kernel has {C_in_k}"
    
    H_out = H - K_h + 1
    W_out = W - K_w + 1
    
    # Unfold input into column matrix: (B, C_in * K_h * K_w, H_out * W_out)
    x_unfold = F.unfold(x, kernel_size=(K_h, K_w))  # (B, C_in*K_h*K_w, L)
    
    # Reshape weight to (C_out, C_in * K_h * K_w)
    w_flat = weight.view(C_out, -1)  # (C_out, C_in*K_h*K_w)
    
    # Matrix multiply: (C_out, C_in*K_h*K_w) @ (B, C_in*K_h*K_w, L) -> (B, C_out, L)
    out = torch.einsum("oi,bil->bol", w_flat, x_unfold)  # (B, C_out, H_out*W_out)
    
    if bias is not None:
        out = out + bias.view(1, -1, 1)  # (B, C_out, H_out*W_out)
    
    return out.view(B, C_out, H_out, W_out)  # (B, C_out, H_out, W_out)


# Test implementation
torch.manual_seed(42)
x_test = torch.randn(2, 3, 8, 8)  # (B, C_in, H, W)
w_test = torch.randn(4, 3, 3, 3)  # (C_out, C_in, K_h, K_w)
b_test = torch.randn(4)            # (C_out,)

out_custom = conv2d_multichannel(x_test, w_test, b_test)  # (2, 4, 6, 6)
out_ref = F.conv2d(x_test, w_test, b_test)                # (2, 4, 6, 6)
assert torch.allclose(out_custom, out_ref, atol=1e-4), f"Max error: {(out_custom - out_ref).abs().max()}"
print(f"Output shape: {out_custom.shape}")  # (2, 4, 6, 6)
print("Multi-channel conv matches F.conv2d ✓")

Now let's put our convolution to work. Hand-crafted kernels like Sobel and Laplacian detect edges — this is what early convolutional layers learn to do automatically.

In [ ]:
# Apply edge detection kernels using our custom conv2d_multichannel

# Create a synthetic test image: white square on black background
img = torch.zeros(1, 1, 64, 64)  # (B, C, H, W)
img[0, 0, 16:48, 16:48] = 1.0    # White square

# Define edge detection kernels
sobel_x = torch.tensor([[-1., 0., 1.],
                         [-2., 0., 2.],
                         [-1., 0., 1.]]).view(1, 1, 3, 3)  # (1, 1, 3, 3)

sobel_y = torch.tensor([[-1., -2., -1.],
                         [ 0.,  0.,  0.],
                         [ 1.,  2.,  1.]]).view(1, 1, 3, 3)  # (1, 1, 3, 3)

laplacian = torch.tensor([[ 0., 1., 0.],
                           [ 1., -4., 1.],
                           [ 0., 1., 0.]]).view(1, 1, 3, 3)  # (1, 1, 3, 3)

# Apply with our custom implementation
edges_x = conv2d_multichannel(img, sobel_x).squeeze()    # (62, 62)
edges_y = conv2d_multichannel(img, sobel_y).squeeze()    # (62, 62)
edges_lap = conv2d_multichannel(img, laplacian).squeeze() # (62, 62)
edges_mag = torch.sqrt(edges_x**2 + edges_y**2)          # (62, 62) -- Sobel magnitude

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
titles = ["Original", "Sobel X (vertical)", "Sobel Y (horizontal)", "Sobel magnitude", "Laplacian"]
images = [img.squeeze(), edges_x, edges_y, edges_mag, edges_lap]

for ax, title, im in zip(axes, titles, images):
    ax.imshow(im.detach().numpy(), cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("Edge Detection with Hand-Crafted Convolution Kernels", fontsize=13)
plt.tight_layout()
plt.show()
print("These are the kinds of features early conv layers learn automatically during training.")

---
## 2.2 — Padding, Stride, and Dilation

These three parameters control the spatial dimensions of the output. Understanding them is essential for designing architectures where tensor shapes need to match up precisely — especially U-Nets, where the decoder must mirror the encoder's spatial dimensions.

### Padding

Padding adds zeros around the input border. Without it, a 3x3 kernel on a 32x32 input produces a 30x30 output — you lose pixels every layer. With `padding=1`, the input is surrounded by one row/column of zeros on each side, giving a 34x34 effective input, and the output stays 32x32.

### Stride

Stride controls how far the kernel moves between positions. `stride=1` means it moves one pixel at a time; `stride=2` means it skips every other position, **halving the spatial dimensions**. This is exactly how U-Net encoders downsample — no pooling layers needed.

### Dilation

Dilation inserts gaps between kernel elements, expanding the receptive field without adding parameters. A 3x3 kernel with `dilation=2` covers a 5x5 area but still has only 9 weights.

### Output size formula

$$H_{\text{out}} = \left\lfloor \frac{H_{\text{in}} + 2 \cdot \text{padding} - \text{dilation} \cdot (\text{kernel} - 1) - 1}{\text{stride}} \right\rfloor + 1$$

Let's make this concrete: a 32x32 input with a 3x3 kernel, `padding=1`, `stride=2` gives $\lfloor(32 + 2 - 3)/2\rfloor + 1 = 16$. That's the standard encoder downsample in U-Nets — one strided convolution halves the resolution.

### Worked example: output size calculations

Here's a helper function that implements the formula, plus a table showing common configurations you'll encounter.

In [ ]:
def output_size(
    input_size: int,
    kernel_size: int,
    padding: int = 0,
    stride: int = 1,
    dilation: int = 1,
) -> int:
    """Compute the output spatial dimension for a convolution."""
    return (input_size + 2 * padding - dilation * (kernel_size - 1) - 1) // stride + 1


# Demonstrate various configurations
configs = [
    {"input_size": 32, "kernel_size": 3, "padding": 0, "stride": 1, "dilation": 1},
    {"input_size": 32, "kernel_size": 3, "padding": 1, "stride": 1, "dilation": 1},  # "same"
    {"input_size": 32, "kernel_size": 3, "padding": 1, "stride": 2, "dilation": 1},  # halve
    {"input_size": 32, "kernel_size": 3, "padding": 2, "stride": 1, "dilation": 2},  # dilated
    {"input_size": 32, "kernel_size": 5, "padding": 2, "stride": 1, "dilation": 1},  # 5x5 same
]

print(f"{'Input':>6} {'Kernel':>7} {'Pad':>4} {'Stride':>7} {'Dilation':>9} {'Output':>7}")
print("-" * 48)
for c in configs:
    out = output_size(**c)
    print(f"{c['input_size']:>6} {c['kernel_size']:>7} {c['padding']:>4} "
          f"{c['stride']:>7} {c['dilation']:>9} {out:>7}")

In [ ]:
# Demonstrate stride=2 halving -- the standard U-Net encoder downsample
torch.manual_seed(42)
x = torch.randn(1, 16, 64, 64)  # (B, C, H, W)

# Stride-2 conv: halves spatial dims, doubles channels (common pattern)
conv_down = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1)
out = conv_down(x)  # (1, 32, 32, 32)
print(f"Input:  {x.shape}  ->  Output: {out.shape}")
print(f"Spatial: 64 -> {out.shape[-1]}  (halved)")
print(f"Channels: 16 -> {out.shape[1]} (doubled)")
print("This is exactly what happens at each U-Net encoder stage.")

### Exercise 2.2: Compute "same" padding

**Implement `compute_same_padding`** that returns the padding needed so that `output_size = ceil(input_size / stride)`.

- For `stride=1`, the output has the same spatial size as the input
- For `stride=2`, the output is exactly half the input size
- Account for dilation by computing the **effective kernel size**: `dilation * (kernel_size - 1) + 1`

In [ ]:
# YOUR CODE HERE — Exercise 2.2

def compute_same_padding(
    kernel_size: int,
    stride: int = 1,
    dilation: int = 1,
) -> int:
    """Compute padding needed so output_size = ceil(input_size / stride).
    
    Args:
        kernel_size: Size of the convolution kernel.
        stride: Convolution stride.
        dilation: Convolution dilation.
    
    Returns:
        Padding value (assumes symmetric padding).
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
test_cases = [
    (3, 1, 1, 1),   # kernel=3, stride=1, dilation=1 -> padding=1
    (5, 1, 1, 2),   # kernel=5, stride=1, dilation=1 -> padding=2
    (3, 1, 2, 2),   # kernel=3, stride=1, dilation=2 -> padding=2
    (3, 2, 1, 1),   # kernel=3, stride=2, dilation=1 -> padding=1
]
for k, s, d, expected in test_cases:
    result = compute_same_padding(k, stride=s, dilation=d)
    assert result == expected, f"kernel={k}, stride={s}, dilation={d}: expected {expected}, got {result}"
    # Verify with actual convolution
    inp = torch.randn(1, 1, 32, 32)
    out = F.conv2d(inp, torch.randn(1, 1, k, k), padding=result, stride=s, dilation=d)
    expected_spatial = (32 + s - 1) // s
    assert out.shape[-1] == expected_spatial, f"Expected output {expected_spatial}, got {out.shape[-1]}"
    print(f"kernel={k}, stride={s}, dilation={d} -> padding={result}, output_size={out.shape[-1]} ✓")
print("All same-padding tests pass ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def compute_same_padding(
    kernel_size: int,
    stride: int = 1,
    dilation: int = 1,
) -> int:
    """Compute padding needed so output_size = ceil(input_size / stride)."""
    # Effective kernel size accounting for dilation
    effective_k = dilation * (kernel_size - 1) + 1
    # For "same" output: padding = (effective_k - 1) // 2
    return (effective_k - 1) // 2


# Verify
test_cases = [
    (3, 1, 1, 1), (5, 1, 1, 2), (3, 1, 2, 2), (3, 2, 1, 1), (7, 1, 1, 3),
]
for k, s, d, expected in test_cases:
    result = compute_same_padding(k, stride=s, dilation=d)
    inp = torch.randn(1, 1, 32, 32)
    out = F.conv2d(inp, torch.randn(1, 1, k, k), padding=result, stride=s, dilation=d)
    expected_spatial = (32 + s - 1) // s
    assert result == expected, f"Expected padding={expected}, got {result}"
    assert out.shape[-1] == expected_spatial, f"Expected output {expected_spatial}, got {out.shape[-1]}"
    print(f"kernel={k}, stride={s}, dilation={d} -> padding={result}, output_size={out.shape[-1]} ✓")
print("All same-padding tests pass ✓")

---
## 2.3 — Transposed Convolutions (Upsampling)

The U-Net encoder reduces spatial resolution step by step (32x32 → 16x16 → 8x8). The decoder needs to go the other direction — increasing resolution back up. There are two common approaches:

- **Transposed convolution** — learnable upsampling
- **Bilinear interpolation + regular conv** — often preferred in practice

### What a transposed convolution actually does

A transposed convolution is **not** the inverse of convolution. Mechanically, it inserts zeros between input elements, then applies a regular convolution. The output size formula is the "reverse" of the standard one:

$$H_{\text{out}} = (H_{\text{in}} - 1) \cdot \text{stride} - 2 \cdot \text{padding} + \text{dilation} \cdot (\text{kernel} - 1) + \text{output\_padding} + 1$$

For example, a 16x16 feature map with `stride=2`, `kernel_size=4`, `padding=1` produces a 32x32 output — doubling the spatial resolution.

### Checkerboard artifacts

Here's the catch: when `kernel_size` is not divisible by `stride`, different output positions receive contributions from different numbers of input elements. This creates a grid-like pattern called **checkerboard artifacts**.

The preferred alternative in many diffusion models is `F.interpolate` (nearest or bilinear) followed by a regular convolution. Let's see both approaches.

### Worked example: transposed convolution mechanics

In [ ]:
# Demonstrate the zero-insertion mechanism of transposed conv
torch.manual_seed(42)

x_small = torch.tensor([[1., 2.],
                         [3., 4.]]).view(1, 1, 2, 2)  # (1, 1, 2, 2)

# Transposed conv with stride=2: upsamples 2x2 -> 4x4
tconv = nn.ConvTranspose2d(1, 1, kernel_size=3, stride=2, padding=1, output_padding=1, bias=False)

# Set identity-like kernel for visualization
with torch.no_grad():
    tconv.weight.fill_(0)
    tconv.weight[0, 0, 1, 1] = 1.0  # Center element only

out_tconv = tconv(x_small)  # (1, 1, 4, 4)
print("Input (2x2):")
print(x_small.squeeze())
print(f"\nTransposed conv output (4x4) with identity-center kernel:")
print(out_tconv.squeeze())
print(f"\nInput shape: {x_small.shape} -> Output shape: {out_tconv.shape}")

In [ ]:
# Demonstrate checkerboard artifacts
torch.manual_seed(42)

x_feat = torch.randn(1, 1, 8, 8)  # (1, 1, 8, 8)

# Bad: kernel_size=3, stride=2 -> checkerboard
tconv_bad = nn.ConvTranspose2d(1, 1, kernel_size=3, stride=2, padding=1, output_padding=1)

# Good: kernel_size=4, stride=2 (divisible) -> less artifact
tconv_good = nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1)

# Alternative: interpolate + conv (preferred in practice)
conv_after_interp = nn.Conv2d(1, 1, kernel_size=3, padding=1)

out_bad = tconv_bad(x_feat)  # (1, 1, 16, 16)
out_good = tconv_good(x_feat)  # (1, 1, 16, 16)
out_interp = conv_after_interp(
    F.interpolate(x_feat, scale_factor=2, mode="nearest")  # (1, 1, 16, 16)
)  # (1, 1, 16, 16)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ["Input (8x8)", "TransConv k=3,s=2\n(checkerboard)",
          "TransConv k=4,s=2\n(less artifact)", "Interpolate + Conv\n(preferred)"]
images = [x_feat.squeeze(), out_bad.squeeze(), out_good.squeeze(), out_interp.squeeze()]

for ax, title, im in zip(axes, titles, images):
    ax.imshow(im.detach().numpy(), cmap="viridis")
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("Transposed Convolution: Checkerboard Artifacts", fontsize=13)
plt.tight_layout()
plt.show()
print("The interpolate + conv approach avoids checkerboard artifacts entirely.")

### Exercise 2.3: Compare two upsampling approaches

**Implement two upsampling modules** that both map `(B, in_channels, H, W)` → `(B, out_channels, 2H, 2W)`:

- `UpsampleTransConv` — uses a transposed convolution (`kernel_size=4, stride=2, padding=1` for clean 2x upsampling)
- `UpsampleInterpConv` — uses `F.interpolate(scale_factor=2, mode="nearest")` followed by a 3x3 conv

In [ ]:
# YOUR CODE HERE — Exercise 2.3

class UpsampleTransConv(nn.Module):
    """Upsample 2x using transposed convolution."""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        # ===================== YOUR CODE HERE =====================
        pass  # Replace with your implementation
        # ====================== END YOUR CODE ======================

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ===================== YOUR CODE HERE =====================
        pass  # Replace with your implementation
        # ====================== END YOUR CODE ======================


class UpsampleInterpConv(nn.Module):
    """Upsample 2x using nearest interpolation + 3x3 conv."""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        # ===================== YOUR CODE HERE =====================
        pass  # Replace with your implementation
        # ====================== END YOUR CODE ======================

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ===================== YOUR CODE HERE =====================
        pass  # Replace with your implementation
        # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
torch.manual_seed(42)
x_test = torch.randn(2, 32, 16, 16)  # (B, C, H, W)
up1 = UpsampleTransConv(32, 16)
up2 = UpsampleInterpConv(32, 16)

out1 = up1(x_test)
out2 = up2(x_test)
assert out1 is not None, "UpsampleTransConv returned None"
assert out2 is not None, "UpsampleInterpConv returned None"
assert out1.shape == (2, 16, 32, 32), f"TransConv: expected (2, 16, 32, 32) but got {out1.shape} — check kernel_size/stride/padding"
assert out2.shape == (2, 16, 32, 32), f"InterpConv: expected (2, 16, 32, 32) but got {out2.shape}"
print(f"TransConv upsample:   {x_test.shape} -> {out1.shape} ✓")
print(f"Interp+Conv upsample: {x_test.shape} -> {out2.shape} ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class UpsampleTransConv(nn.Module):
    """Upsample 2x using transposed convolution."""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        # kernel=4, stride=2, padding=1 -> doubles spatial dims cleanly
        self.tconv = nn.ConvTranspose2d(
            in_channels, out_channels, kernel_size=4, stride=2, padding=1
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.tconv(x)  # (B, C_out, 2*H, 2*W)


class UpsampleInterpConv(nn.Module):
    """Upsample 2x using nearest interpolation + 3x3 conv."""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.interpolate(x, scale_factor=2, mode="nearest")  # (B, C, 2*H, 2*W)
        return self.conv(x)  # (B, C_out, 2*H, 2*W)


# Test both
torch.manual_seed(42)
x_test = torch.randn(2, 32, 16, 16)  # (B, C, H, W)
up1 = UpsampleTransConv(32, 16)
up2 = UpsampleInterpConv(32, 16)

out1 = up1(x_test)  # (2, 16, 32, 32)
out2 = up2(x_test)  # (2, 16, 32, 32)

print(f"TransConv upsample:   {x_test.shape} -> {out1.shape} ✓")
print(f"Interp+Conv upsample: {x_test.shape} -> {out2.shape} ✓")

# Compare parameter counts
p1 = sum(p.numel() for p in up1.parameters())
p2 = sum(p.numel() for p in up2.parameters())
print(f"\nTransConv params:   {p1:,}")
print(f"Interp+Conv params: {p2:,}")
print(f"\nInterp+Conv is preferred in many diffusion models (no checkerboard).")

---
## 2.4 — Depthwise Separable Convolutions

A standard convolution mixes spatial and channel information in one step. A **depthwise separable convolution** splits this into two cheaper steps:

### Depthwise convolution

Each input channel is convolved independently with its own 2D kernel (`groups=C_in`). This captures spatial patterns without mixing channels.

### Pointwise convolution

A 1x1 convolution mixes information across channels. Together, the two steps approximate a standard convolution at roughly **1/9th the cost** for 3x3 kernels.

| Type | Parameters |
|------|-----------|
| **Standard** 3x3 | C_in × C_out × 9 |
| **Separable** (DW+PW) | C_in × 9 + C_in × C_out |

### Worked example: depthwise separable conv

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    """Depthwise separable convolution = depthwise + pointwise."""
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, padding: int = 1):
        super().__init__()
        # Depthwise: each channel convolved independently (groups=in_channels)
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=kernel_size,
            padding=padding, groups=in_channels, bias=False
        )
        # Pointwise: 1x1 conv to mix channels
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)   # (B, C_in, H, W)
        x = self.pointwise(x)   # (B, C_out, H, W)
        return x


# Compare parameter counts
C_in, C_out = 64, 128
standard = nn.Conv2d(C_in, C_out, kernel_size=3, padding=1)
separable = DepthwiseSeparableConv(C_in, C_out)

params_std = sum(p.numel() for p in standard.parameters())
params_sep = sum(p.numel() for p in separable.parameters())

print(f"Standard conv params:  {params_std:>8,}")
print(f"Separable conv params: {params_sep:>8,}")
print(f"Ratio: {params_sep / params_std:.3f} ({params_std / params_sep:.1f}x fewer)")

# Verify same output shape
torch.manual_seed(42)
x_test = torch.randn(2, C_in, 16, 16)  # (B, C_in, H, W)
print(f"\nStandard output shape:  {standard(x_test).shape}")   # (2, 128, 16, 16)
print(f"Separable output shape: {separable(x_test).shape}")    # (2, 128, 16, 16)

### Exercise 2.4: Replace standard with separable, compare speed

**Build two small 3-layer ConvNets** — one with standard convolutions, one with `DepthwiseSeparableConv` — and compare parameter counts and forward pass speed.

- Architecture: 3 conv layers mapping 3→32→64→128 channels, each with `kernel=3, padding=1`
- Add ReLU activations between layers
- Use the `use_separable` flag to switch between standard and separable

In [ ]:
# YOUR CODE HERE — Exercise 2.4

class SmallConvNet(nn.Module):
    """3-layer ConvNet with standard or separable convolutions."""
    def __init__(self, use_separable: bool = False):
        super().__init__()
        # Build 3 conv layers: 3->32->64->128, each with kernel=3, padding=1
        # If use_separable, use DepthwiseSeparableConv instead of nn.Conv2d
        # Add ReLU activations between layers
        # ===================== YOUR CODE HERE =====================
        pass  # Replace with your implementation
        # ====================== END YOUR CODE ======================

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ===================== YOUR CODE HERE =====================
        pass  # Replace with your implementation
        # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
torch.manual_seed(42)
x_test = torch.randn(2, 3, 16, 16)

net_std = SmallConvNet(use_separable=False)
net_sep = SmallConvNet(use_separable=True)

out_std = net_std(x_test)
out_sep = net_sep(x_test)
assert out_std is not None, "Standard net returned None"
assert out_sep is not None, "Separable net returned None"
assert out_std.shape == (2, 128, 16, 16), f"Standard: expected (2, 128, 16, 16) but got {out_std.shape}"
assert out_sep.shape == (2, 128, 16, 16), f"Separable: expected (2, 128, 16, 16) but got {out_sep.shape}"

params_std = sum(p.numel() for p in net_std.parameters())
params_sep = sum(p.numel() for p in net_sep.parameters())
assert params_sep < params_std, f"Separable ({params_sep}) should have fewer params than standard ({params_std})"
print(f"Standard params:  {params_std:,}")
print(f"Separable params: {params_sep:,}")
print(f"Output shapes match: {out_std.shape} ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class SmallConvNet(nn.Module):
    """3-layer ConvNet with standard or separable convolutions."""
    def __init__(self, use_separable: bool = False):
        super().__init__()
        ConvBlock = DepthwiseSeparableConv if use_separable else lambda c_in, c_out, **kw: nn.Conv2d(c_in, c_out, kernel_size=3, padding=1)
        
        self.layers = nn.Sequential(
            ConvBlock(3, 32),
            nn.ReLU(),
            ConvBlock(32, 64),
            nn.ReLU(),
            ConvBlock(64, 128),
            nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)  # (B, 128, H, W)


net_std = SmallConvNet(use_separable=False)
net_sep = SmallConvNet(use_separable=True)

params_std = sum(p.numel() for p in net_std.parameters())
params_sep = sum(p.numel() for p in net_sep.parameters())
print(f"Standard params:  {params_std:,}")
print(f"Separable params: {params_sep:,}")
print(f"Reduction: {params_sep / params_std:.3f}x")

# Speed comparison
x_bench = torch.randn(8, 3, 64, 64)

# Warm up
for _ in range(3):
    _ = net_std(x_bench)
    _ = net_sep(x_bench)

n_trials = 20
t0 = time.time()
for _ in range(n_trials):
    _ = net_std(x_bench)
t_std = (time.time() - t0) / n_trials

t0 = time.time()
for _ in range(n_trials):
    _ = net_sep(x_bench)
t_sep = (time.time() - t0) / n_trials

print(f"\nStandard forward:  {t_std*1000:.2f} ms")
print(f"Separable forward: {t_sep*1000:.2f} ms")
print(f"Speedup: {t_std/t_sep:.2f}x")

---
## 2.5 — Normalization Layers

Without normalization, training deep networks is a nightmare. Activations drift and explode as they pass through layers, making optimization unstable. Normalization fixes this by re-centering and re-scaling activations:

$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}, \quad y = \gamma \hat{x} + \beta$$

where $\mu$ and $\sigma^2$ are computed over some subset of dimensions, and $\gamma$, $\beta$ are learnable scale/shift parameters.

The four common variants differ only in **which dimensions** they average over:

| Variant | Averages over | Batch-size dependent? |
|---------|--------------|----------------------|
| **BatchNorm** | batch, height, width | Yes |
| **LayerNorm** | channels, height, width | No |
| **InstanceNorm** | height, width (per channel) | No |
| **GroupNorm** | channels within each group, height, width | No |

**Why diffusion models use GroupNorm:** Diffusion training often uses small batch sizes (2-8 per GPU). BatchNorm's statistics become noisy with small batches. GroupNorm computes statistics per-sample, so it works reliably regardless of batch size. The standard choice is 32 groups.

### Worked example: BatchNorm from scratch

Let's implement batch normalization by hand to see exactly what's happening inside.

In [ ]:
class BatchNorm2dFromScratch(nn.Module):
    """Batch normalization for (B, C, H, W) tensors, implemented from scratch."""
    
    def __init__(self, num_features: int, eps: float = 1e-5, momentum: float = 0.1):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        
        # Learnable affine parameters
        self.gamma = nn.Parameter(torch.ones(num_features))   # (C,)
        self.beta = nn.Parameter(torch.zeros(num_features))   # (C,)
        
        # Running statistics (not parameters -- not updated by optimizer)
        self.register_buffer("running_mean", torch.zeros(num_features))  # (C,)
        self.register_buffer("running_var", torch.ones(num_features))    # (C,)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.
        
        Args:
            x: Input of shape (B, C, H, W)
        Returns:
            Normalized output of shape (B, C, H, W)
        """
        if self.training:
            # Compute batch statistics over (B, H, W) for each channel
            mean = x.mean(dim=(0, 2, 3))  # (C,)
            var = x.var(dim=(0, 2, 3), unbiased=False)  # (C,)
            
            # Update running stats (EMA)
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var
        else:
            mean = self.running_mean  # (C,)
            var = self.running_var    # (C,)
        
        # Normalize: reshape stats to (1, C, 1, 1) for broadcasting
        mean = mean.view(1, -1, 1, 1)   # (1, C, 1, 1)
        var = var.view(1, -1, 1, 1)      # (1, C, 1, 1)
        gamma = self.gamma.view(1, -1, 1, 1)  # (1, C, 1, 1)
        beta = self.beta.view(1, -1, 1, 1)    # (1, C, 1, 1)
        
        x_hat = (x - mean) / torch.sqrt(var + self.eps)  # (B, C, H, W)
        return gamma * x_hat + beta  # (B, C, H, W)


# Verify against PyTorch
torch.manual_seed(42)
x = torch.randn(4, 8, 16, 16)  # (B, C, H, W)

bn_ours = BatchNorm2dFromScratch(8)
bn_ref = nn.BatchNorm2d(8)

# Both in training mode
out_ours = bn_ours(x)  # (4, 8, 16, 16)
out_ref = bn_ref(x)    # (4, 8, 16, 16)

print(f"Output shape: {out_ours.shape}")
print(f"Max error: {(out_ours - out_ref).abs().max():.2e}")
assert torch.allclose(out_ours, out_ref, atol=1e-5)
print("BatchNorm from scratch matches nn.BatchNorm2d ✓")

# Verify running stats were updated
print(f"\nRunning mean (first 4 channels): {bn_ours.running_mean[:4].tolist()}")
print(f"Running var  (first 4 channels): {bn_ours.running_var[:4].tolist()}")

### Exercise 2.5: Train with and without BatchNorm

**Build two small ConvNets** (with and without BatchNorm), train on synthetic data, and compare convergence speed.

- Use a 3-layer ConvNet: Conv→(BN)→ReLU→MaxPool, repeated 3 times, then global average pooling → linear classifier
- Create synthetic data: `torch.randn(256, 3, 16, 16)` inputs with random labels
- Train for 200 steps, record losses, and plot both curves

In [ ]:
# YOUR CODE HERE — Exercise 2.5

class ConvClassifier(nn.Module):
    """Simple ConvNet classifier with optional BatchNorm."""
    def __init__(self, use_batchnorm: bool = False, num_classes: int = 10):
        super().__init__()
        # Build: conv1(3->32) -> [bn1] -> relu -> pool
        #        conv2(32->64) -> [bn2] -> relu -> pool
        #        conv3(64->128) -> [bn3] -> relu -> global_avg_pool -> fc(128->10)
        # ===================== YOUR CODE HERE =====================
        pass  # Replace with your implementation
        # ====================== END YOUR CODE ======================
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ===================== YOUR CODE HERE =====================
        pass  # Replace with your implementation
        # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
torch.manual_seed(42)
model_bn = ConvClassifier(use_batchnorm=True)
model_no = ConvClassifier(use_batchnorm=False)
x_test = torch.randn(4, 3, 16, 16)
out_bn = model_bn(x_test)
out_no = model_no(x_test)
assert out_bn is not None, "BatchNorm model returned None"
assert out_bn.shape == (4, 10), f"Expected (4, 10) but got {out_bn.shape}"
assert out_no.shape == (4, 10), f"Expected (4, 10) but got {out_no.shape}"
print(f"Output shape: {out_bn.shape} ✓")
print("Both models forward pass works. Now train them to compare convergence!")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class ConvClassifier(nn.Module):
    """Simple ConvNet classifier with optional BatchNorm."""
    def __init__(self, use_batchnorm: bool = False, num_classes: int = 10):
        super().__init__()
        self.use_bn = use_batchnorm
        
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        
        if use_batchnorm:
            self.bn1 = nn.BatchNorm2d(32)
            self.bn2 = nn.BatchNorm2d(64)
            self.bn3 = nn.BatchNorm2d(128)
        
        self.fc = nn.Linear(128, num_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)                                      # (B, 32, H, W)
        x = self.bn1(x) if self.use_bn else x                 # (B, 32, H, W)
        x = F.relu(x)                                          # (B, 32, H, W)
        x = F.max_pool2d(x, 2)                                 # (B, 32, H/2, W/2)
        
        x = self.conv2(x)                                      # (B, 64, H/2, W/2)
        x = self.bn2(x) if self.use_bn else x                 # (B, 64, H/2, W/2)
        x = F.relu(x)                                          # (B, 64, H/2, W/2)
        x = F.max_pool2d(x, 2)                                 # (B, 64, H/4, W/4)
        
        x = self.conv3(x)                                      # (B, 128, H/4, W/4)
        x = self.bn3(x) if self.use_bn else x                 # (B, 128, H/4, W/4)
        x = F.relu(x)                                          # (B, 128, H/4, W/4)
        
        x = x.mean(dim=(2, 3))                                 # (B, 128) -- global avg pool
        return self.fc(x)                                      # (B, num_classes)


# Train both variants on synthetic data
torch.manual_seed(42)
num_steps = 200
batch_size = 32

X_train = torch.randn(256, 3, 16, 16)  # (N, C, H, W)
y_train = torch.randint(0, 10, (256,))  # (N,)

losses = {"Without BatchNorm": [], "With BatchNorm": []}

for use_bn, label in [(False, "Without BatchNorm"), (True, "With BatchNorm")]:
    torch.manual_seed(42)
    model = ConvClassifier(use_batchnorm=use_bn)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    for step in range(num_steps):
        idx = torch.randint(0, len(X_train), (batch_size,))
        xb, yb = X_train[idx], y_train[idx]
        
        logits = model(xb)  # (B, 10)
        loss = F.cross_entropy(logits, yb)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses[label].append(loss.item())

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
for label, loss_list in losses.items():
    ax.plot(loss_list, label=label, alpha=0.8)
ax.set_xlabel("Step")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Convergence: BatchNorm vs No BatchNorm")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("BatchNorm helps the model converge faster and more smoothly.")

---
## 2.6 — GroupNorm: The Diffusion Standard

Now let's look at the normalization layer that diffusion models actually use. GroupNorm divides channels into groups and normalizes within each group independently. For an input of shape `(B, C, H, W)` with `G` groups:

1. Reshape to `(B, G, C//G, H, W)`
2. Compute mean and variance over dimensions `(2, 3, 4)` — within each group
3. Normalize, then apply learnable $\gamma$ and $\beta$

**Concrete example:** With 64 channels and 32 groups, each group contains 2 channels. The statistics are computed over those 2 channels and all spatial positions, independently for each sample.

GroupNorm is actually a generalization of other normalizations:
- **LayerNorm** when `G = 1` (one group containing all channels)
- **InstanceNorm** when `G = C` (each channel is its own group)

### Worked example: GroupNorm from scratch

In [ ]:
class GroupNormFromScratch(nn.Module):
    """Group normalization for (B, C, H, W) tensors, implemented from scratch."""
    
    def __init__(self, num_groups: int, num_channels: int, eps: float = 1e-5):
        super().__init__()
        assert num_channels % num_groups == 0, "num_channels must be divisible by num_groups"
        self.num_groups = num_groups
        self.num_channels = num_channels
        self.eps = eps
        
        self.gamma = nn.Parameter(torch.ones(num_channels))   # (C,)
        self.beta = nn.Parameter(torch.zeros(num_channels))   # (C,)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.
        
        Args:
            x: Input of shape (B, C, H, W)
        Returns:
            Normalized output of shape (B, C, H, W)
        """
        B, C, H, W = x.shape  # (B, C, H, W)
        G = self.num_groups
        
        # Reshape to (B, G, C//G, H, W) -- group channels
        x = x.view(B, G, C // G, H, W)  # (B, G, C//G, H, W)
        
        # Compute mean and var within each group (over channels-in-group, H, W)
        mean = x.mean(dim=(2, 3, 4), keepdim=True)  # (B, G, 1, 1, 1)
        var = x.var(dim=(2, 3, 4), keepdim=True, unbiased=False)  # (B, G, 1, 1, 1)
        
        # Normalize
        x = (x - mean) / torch.sqrt(var + self.eps)  # (B, G, C//G, H, W)
        
        # Reshape back to (B, C, H, W)
        x = x.view(B, C, H, W)  # (B, C, H, W)
        
        # Apply learnable affine
        gamma = self.gamma.view(1, C, 1, 1)  # (1, C, 1, 1)
        beta = self.beta.view(1, C, 1, 1)    # (1, C, 1, 1)
        return gamma * x + beta  # (B, C, H, W)


# Verify against PyTorch
torch.manual_seed(42)
x = torch.randn(4, 32, 8, 8)  # (B, C, H, W)

gn_ours = GroupNormFromScratch(num_groups=8, num_channels=32)
gn_ref = nn.GroupNorm(num_groups=8, num_channels=32)

out_ours = gn_ours(x)  # (4, 32, 8, 8)
out_ref = gn_ref(x)    # (4, 32, 8, 8)

print(f"Output shape: {out_ours.shape}")
print(f"Max error: {(out_ours - out_ref).abs().max():.2e}")
assert torch.allclose(out_ours, out_ref, atol=1e-5)
print("GroupNorm from scratch matches nn.GroupNorm ✓")

Let's see why batch-size independence matters in practice.

In [ ]:
# Demonstrate batch-size independence (GroupNorm vs BatchNorm)
torch.manual_seed(42)
x_single = torch.randn(1, 32, 8, 8)  # (1, C, H, W) -- single sample
x_batch = torch.cat([x_single, torch.randn(3, 32, 8, 8)], dim=0)  # (4, C, H, W)

# GroupNorm: same output for x_single[0] regardless of batch
gn = nn.GroupNorm(8, 32)
gn_single = gn(x_single)[0]  # (32, 8, 8)
gn_batch = gn(x_batch)[0]    # (32, 8, 8)

# BatchNorm: DIFFERENT output depending on batch
bn = nn.BatchNorm2d(32)
bn.train()
bn_single = bn(x_single)[0]  # (32, 8, 8)
# Reset running stats to avoid contamination
bn = nn.BatchNorm2d(32)
bn.train()
bn_batch = bn(x_batch)[0]    # (32, 8, 8)

print("GroupNorm difference (same sample, different batch sizes):")
print(f"  Max diff: {(gn_single - gn_batch).abs().max():.2e}")
print("\nBatchNorm difference (same sample, different batch sizes):")
print(f"  Max diff: {(bn_single - bn_batch).abs().max():.4f}")
print("\nGroupNorm output is identical regardless of batch size. BatchNorm is not.")
print("This is why diffusion models use GroupNorm.")

### Exercise 2.6: Implement all four normalizations as one function

**Implement `unified_norm`** that handles Batch, Layer, Instance, and Group normalization with a single function.

The key insight: Layer, Instance, and Group norm all follow the same pattern — reshape to `(B, G, C//G, H, W)`, normalize over `(2, 3, 4)`:

- **LayerNorm:** `G = 1` (one group = all channels)
- **InstanceNorm:** `G = C` (each channel is its own group)
- **GroupNorm:** `G = num_groups`
- **BatchNorm:** special case — averages over `(B, H, W)` per channel

In [ ]:
# YOUR CODE HERE — Exercise 2.6

def unified_norm(
    x: torch.Tensor,
    norm_type: str,
    num_groups: int = 32,
    eps: float = 1e-5,
) -> torch.Tensor:
    """Unified normalization (without learnable parameters, for demonstration).
    
    Args:
        x: Input of shape (B, C, H, W)
        norm_type: One of 'batch', 'layer', 'instance', 'group'
        num_groups: Number of groups (only used when norm_type='group')
        eps: Epsilon for numerical stability
    
    Returns:
        Normalized tensor of shape (B, C, H, W)
    """
    B, C, H, W = x.shape
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
torch.manual_seed(42)
x = torch.randn(4, 32, 8, 8)

# Test LayerNorm
ln = nn.LayerNorm([32, 8, 8], elementwise_affine=False)
out_ours = unified_norm(x, "layer")
assert out_ours is not None, "unified_norm returned None"
assert out_ours.shape == x.shape, f"Expected {x.shape} but got {out_ours.shape}"
assert torch.allclose(out_ours, ln(x), atol=1e-5), "LayerNorm doesn't match — check G=1 case"
print("LayerNorm ✓")

# Test InstanceNorm
inst = nn.InstanceNorm2d(32, affine=False)
out_ours = unified_norm(x, "instance")
assert torch.allclose(out_ours, inst(x), atol=1e-5), "InstanceNorm doesn't match — check G=C case"
print("InstanceNorm ✓")

# Test GroupNorm
gn = nn.GroupNorm(8, 32, affine=False)
out_ours = unified_norm(x, "group", num_groups=8)
assert torch.allclose(out_ours, gn(x), atol=1e-5), "GroupNorm doesn't match"
print("GroupNorm ✓")

# Test BatchNorm
bn = nn.BatchNorm2d(32, affine=False)
bn.train()
out_ours = unified_norm(x, "batch")
assert torch.allclose(out_ours, bn(x), atol=1e-5), "BatchNorm doesn't match — did you average over (0, 2, 3)?"
print("BatchNorm ✓")

print("\nAll four normalizations are the same operation with different grouping axes!")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def unified_norm(
    x: torch.Tensor,
    norm_type: str,
    num_groups: int = 32,
    eps: float = 1e-5,
) -> torch.Tensor:
    """Unified normalization (without learnable parameters, for demonstration)."""
    B, C, H, W = x.shape  # (B, C, H, W)
    
    if norm_type == "batch":
        # Mean/var over (B, H, W) for each channel
        mean = x.mean(dim=(0, 2, 3), keepdim=True)  # (1, C, 1, 1)
        var = x.var(dim=(0, 2, 3), keepdim=True, unbiased=False)  # (1, C, 1, 1)
        return (x - mean) / torch.sqrt(var + eps)  # (B, C, H, W)
    
    # For layer/instance/group, reshape to (B, G, C//G, H, W)
    if norm_type == "layer":
        G = 1  # One group = all channels
    elif norm_type == "instance":
        G = C  # Each channel is its own group
    elif norm_type == "group":
        G = num_groups
    else:
        raise ValueError(f"Unknown norm_type: {norm_type}")
    
    x_grouped = x.view(B, G, C // G, H, W)  # (B, G, C//G, H, W)
    mean = x_grouped.mean(dim=(2, 3, 4), keepdim=True)  # (B, G, 1, 1, 1)
    var = x_grouped.var(dim=(2, 3, 4), keepdim=True, unbiased=False)  # (B, G, 1, 1, 1)
    x_normed = (x_grouped - mean) / torch.sqrt(var + eps)  # (B, G, C//G, H, W)
    return x_normed.view(B, C, H, W)  # (B, C, H, W)


# Verify each against PyTorch
torch.manual_seed(42)
x = torch.randn(4, 32, 8, 8)  # (B, C, H, W)

ln = nn.LayerNorm([32, 8, 8], elementwise_affine=False)
print(f"LayerNorm max error:    {(unified_norm(x, 'layer') - ln(x)).abs().max():.2e} ✓")

inst = nn.InstanceNorm2d(32, affine=False)
print(f"InstanceNorm max error: {(unified_norm(x, 'instance') - inst(x)).abs().max():.2e} ✓")

gn = nn.GroupNorm(8, 32, affine=False)
print(f"GroupNorm max error:    {(unified_norm(x, 'group', num_groups=8) - gn(x)).abs().max():.2e} ✓")

bn = nn.BatchNorm2d(32, affine=False)
bn.train()
print(f"BatchNorm max error:    {(unified_norm(x, 'batch') - bn(x)).abs().max():.2e} ✓")

print("\nAll normalizations are the same operation with different grouping axes.")

---
## 2.7 — Residual Connections

This is arguably the most important architectural idea in deep learning. Without residual connections, networks deeper than ~20 layers perform *worse* than shallower ones — not from overfitting, but because optimization becomes impossibly hard.

### The residual connection: y = F(x) + x

Instead of learning a mapping $H(x)$ directly, learn the **residual** $F(x) = H(x) - x$. The output is $y = F(x) + x$.

Two key benefits:

### Gradient flow

The identity shortcut provides a direct gradient path: $\frac{\partial y}{\partial x} = \frac{\partial F}{\partial x} + I$. Even if $\frac{\partial F}{\partial x}$ is small, the gradient is at least $I$. Gradients can flow through 100+ layers without vanishing.

### Easy identity

If the optimal mapping is close to identity, the network only needs to push $F(x)$ toward zero — much easier than learning the identity explicitly.

### Handling dimension mismatches

When `x` and `F(x)` have different channel counts, the shortcut needs a **1x1 projection convolution**: $y = F(x) + W_s x$. This is exactly what happens in U-Net blocks when the channel count changes between encoder stages.

Diffusion models typically use this ordering: **Conv → GroupNorm → SiLU → Conv → GroupNorm + shortcut**.

*Reference: [Deep Residual Learning](https://arxiv.org/abs/1512.03385) — He et al. 2015*

### Worked example: ResBlock with gradient analysis

Let's build a ResBlock and compare gradient flow against an identical architecture without the skip connection.

In [ ]:
class ResBlock(nn.Module):
    """Residual block: Conv -> GroupNorm -> SiLU -> Conv -> GroupNorm + shortcut.
    
    This is the building block of diffusion U-Nets.
    """
    def __init__(self, channels: int, num_groups: int = 32):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups, channels)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x                          # (B, C, H, W) -- identity shortcut
        x = self.conv1(x)                     # (B, C, H, W)
        x = self.gn1(x)                       # (B, C, H, W)
        x = F.silu(x)                         # (B, C, H, W)
        x = self.conv2(x)                     # (B, C, H, W)
        x = self.gn2(x)                       # (B, C, H, W)
        return x + residual                   # (B, C, H, W)


class PlainBlock(nn.Module):
    """Same architecture but WITHOUT the residual connection."""
    def __init__(self, channels: int, num_groups: int = 32):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups, channels)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)                     # (B, C, H, W)
        x = self.gn1(x)                       # (B, C, H, W)
        x = F.silu(x)                         # (B, C, H, W)
        x = self.conv2(x)                     # (B, C, H, W)
        x = self.gn2(x)                       # (B, C, H, W)
        return x                              # (B, C, H, W) -- no shortcut!


# Compare gradient magnitudes through deep stacks
torch.manual_seed(42)
depth = 20
channels = 64  # Must be divisible by 32 (num_groups)

res_blocks = nn.Sequential(*[ResBlock(channels) for _ in range(depth)])
plain_blocks = nn.Sequential(*[PlainBlock(channels) for _ in range(depth)])

x = torch.randn(1, channels, 16, 16, requires_grad=True)  # (1, 64, 16, 16)

# Forward + backward through residual network
out_res = res_blocks(x)  # (1, 64, 16, 16)
loss_res = out_res.sum()
loss_res.backward()
grad_res = x.grad.norm().item()

x.grad = None  # Reset gradient

# Forward + backward through plain network
out_plain = plain_blocks(x)  # (1, 64, 16, 16)
loss_plain = out_plain.sum()
loss_plain.backward()
grad_plain = x.grad.norm().item()

print(f"Depth: {depth} blocks ({depth * 2} conv layers)")
print(f"Gradient norm at input (residual):  {grad_res:.4f}")
print(f"Gradient norm at input (plain):     {grad_plain:.6f}")
print(f"Ratio (residual / plain): {grad_res / (grad_plain + 1e-12):.1f}x stronger")
print("\nResidual connections preserve gradient flow through deep networks.")

### Exercise 2.7: ResBlock with optional dimension change

**Implement `ResBlockWithProjection`** — the exact building block used in diffusion U-Nets.

- Architecture: Conv 3x3 → GroupNorm → SiLU → Conv 3x3 → GroupNorm + shortcut
- When `in_channels != out_channels`, the shortcut uses a **1x1 conv** to match dimensions
- When `in_channels == out_channels`, the shortcut is just `nn.Identity()`

In [ ]:
# YOUR CODE HERE — Exercise 2.7

class ResBlockWithProjection(nn.Module):
    """Residual block with optional 1x1 projection for channel changes."""
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        num_groups: int = 32,
    ):
        super().__init__()
        # ===================== YOUR CODE HERE =====================
        pass  # Replace with your implementation
        # ====================== END YOUR CODE ======================

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ===================== YOUR CODE HERE =====================
        pass  # Replace with your implementation
        # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
torch.manual_seed(42)
x1 = torch.randn(2, 64, 16, 16)  # (B, C, H, W)

# Same channels -- should use identity shortcut
block_same = ResBlockWithProjection(64, 64)
out_same = block_same(x1)
assert out_same is not None, "ResBlockWithProjection returned None"
assert out_same.shape == (2, 64, 16, 16), f"Same channels: expected (2, 64, 16, 16), got {out_same.shape}"
print(f"Same channels: {x1.shape} -> {out_same.shape} ✓")

# Different channels -- should use 1x1 projection
block_diff = ResBlockWithProjection(64, 128)
out_diff = block_diff(x1)
assert out_diff.shape == (2, 128, 16, 16), f"Diff channels: expected (2, 128, 16, 16), got {out_diff.shape}"
print(f"Diff channels: {x1.shape} -> {out_diff.shape} ✓")

# Verify gradient flows
out_diff.sum().backward()
assert x1.grad is not None, "No gradient flowing back — check your forward pass"
print("Gradient flows through the block ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class ResBlockWithProjection(nn.Module):
    """Residual block with optional 1x1 projection for channel changes.
    
    This is the exact block used in diffusion U-Nets.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        num_groups: int = 32,
    ):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        
        # 1x1 projection shortcut if channels change
        if in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        else:
            self.shortcut = nn.Identity()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.shortcut(x)           # (B, C_out, H, W)
        x = self.conv1(x)                     # (B, C_out, H, W)
        x = self.gn1(x)                       # (B, C_out, H, W)
        x = F.silu(x)                         # (B, C_out, H, W)
        x = self.conv2(x)                     # (B, C_out, H, W)
        x = self.gn2(x)                       # (B, C_out, H, W)
        return x + residual                   # (B, C_out, H, W)


# Test both cases
torch.manual_seed(42)
x1 = torch.randn(2, 64, 16, 16)  # (B, C, H, W)

# Same channels -- identity shortcut
block_same = ResBlockWithProjection(64, 64)
out_same = block_same(x1)  # (2, 64, 16, 16)
print(f"Same channels: {x1.shape} -> {out_same.shape}")
print(f"  Shortcut type: {type(block_same.shortcut).__name__}")

# Different channels -- 1x1 projection
block_diff = ResBlockWithProjection(64, 128)
out_diff = block_diff(x1)  # (2, 128, 16, 16)
print(f"Diff channels: {x1.shape} -> {out_diff.shape}")
print(f"  Shortcut type: {type(block_diff.shortcut).__name__}")

# Count parameters
params_same = sum(p.numel() for p in block_same.parameters())
params_diff = sum(p.numel() for p in block_diff.parameters())
print(f"\nParams (same channels):  {params_same:,}")
print(f"Params (diff channels):  {params_diff:,}")
print(f"Extra from 1x1 projection: {params_diff - params_same:,}")

---
## Capstone: CIFAR-10 ConvNet with Your Custom Building Blocks

Time to put everything together! You'll build a small ConvNet classifier using the exact same components found in diffusion U-Net encoders:

- **Conv2d layers** for feature extraction
- **GroupNorm** (not BatchNorm)
- **Residual connections** with 1x1 projection when channels change
- **SiLU activation** (not ReLU)
- **Stride-2 convolution** for downsampling (not pooling)

Target: **>70% accuracy** on CIFAR-10 test set in 15 epochs. Here's the suggested architecture:

```
Input (3, 32, 32)
  -> Conv2d(3, 64, 3, padding=1)   -> (64, 32, 32)
  -> ResBlock(64, 64)              -> (64, 32, 32)
  -> Conv2d(64, 128, 3, stride=2)  -> (128, 16, 16)   # Downsample
  -> ResBlock(128, 128)            -> (128, 16, 16)
  -> Conv2d(128, 256, 3, stride=2) -> (256, 8, 8)      # Downsample
  -> ResBlock(256, 256)            -> (256, 8, 8)
  -> Global Average Pool           -> (256,)
  -> Linear(256, 10)               -> (10,)
```

In [ ]:
# YOUR CODE HERE — Capstone Exercise

class CapstoneCIFAR10Net(nn.Module):
    """Small ConvNet for CIFAR-10 using diffusion-model building blocks."""
    def __init__(self, num_classes: int = 10):
        super().__init__()
        # Use ResBlockWithProjection from Exercise 2.7
        # Use GroupNorm, SiLU, stride-2 conv for downsampling
        # ===================== YOUR CODE HERE =====================
        pass  # Replace with your implementation
        # ====================== END YOUR CODE ======================
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ===================== YOUR CODE HERE =====================
        pass  # Replace with your implementation
        # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
torch.manual_seed(42)
model = CapstoneCIFAR10Net()
test_input = torch.randn(2, 3, 32, 32)
test_output = model(test_input)
assert test_output is not None, "Model returned None"
assert test_output.shape == (2, 10), f"Expected (2, 10) but got {test_output.shape}"
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {test_output.shape} ✓")
print("Architecture looks good! Now let's train it.")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

import torchvision
import torchvision.transforms as transforms


class CapstoneCIFAR10Net(nn.Module):
    """Small ConvNet for CIFAR-10 using diffusion-model building blocks.
    
    Uses GroupNorm, SiLU, residual connections, and stride-2 downsampling --
    the same components found in diffusion U-Net encoders.
    """
    def __init__(self, num_classes: int = 10):
        super().__init__()
        
        # Initial projection
        self.stem = nn.Conv2d(3, 64, 3, padding=1)                # (B, 64, 32, 32)
        
        # Stage 1: 32x32 resolution
        self.res1 = ResBlockWithProjection(64, 64, num_groups=32)  # (B, 64, 32, 32)
        
        # Downsample 32x32 -> 16x16
        self.down1 = nn.Conv2d(64, 128, 3, stride=2, padding=1)   # (B, 128, 16, 16)
        
        # Stage 2: 16x16 resolution
        self.res2 = ResBlockWithProjection(128, 128, num_groups=32)  # (B, 128, 16, 16)
        
        # Downsample 16x16 -> 8x8
        self.down2 = nn.Conv2d(128, 256, 3, stride=2, padding=1)  # (B, 256, 8, 8)
        
        # Stage 3: 8x8 resolution
        self.res3 = ResBlockWithProjection(256, 256, num_groups=32)  # (B, 256, 8, 8)
        
        # Classification head
        self.norm_out = nn.GroupNorm(32, 256)
        self.fc = nn.Linear(256, num_classes)                     # (B, 10)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.silu(self.stem(x))      # (B, 64, 32, 32)
        x = self.res1(x)              # (B, 64, 32, 32)
        x = F.silu(self.down1(x))     # (B, 128, 16, 16)
        x = self.res2(x)              # (B, 128, 16, 16)
        x = F.silu(self.down2(x))     # (B, 256, 8, 8)
        x = self.res3(x)              # (B, 256, 8, 8)
        x = F.silu(self.norm_out(x))  # (B, 256, 8, 8)
        x = x.mean(dim=(2, 3))        # (B, 256) -- global average pool
        return self.fc(x)             # (B, 10)


model = CapstoneCIFAR10Net().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

# Verify output shape
test_input = torch.randn(2, 3, 32, 32).to(device)
test_output = model(test_input)  # (2, 10)
print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {test_output.shape} ✓")

> **macOS note:** If you encounter multiprocessing errors, change `num_workers=2` to `num_workers=0` in the DataLoader calls below.

In [ ]:
# ✅ SOLUTION (continued) — Data loading

# Data augmentation (standard for CIFAR-10)
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

trainset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_train
)
testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_test
)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

print(f"Training samples: {len(trainset):,}")
print(f"Test samples:     {len(testset):,}")

In [ ]:
# ✅ SOLUTION (continued) — Training loop

torch.manual_seed(42)
model = CapstoneCIFAR10Net().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

num_epochs = 15
train_losses = []
test_accs = []

for epoch in range(num_epochs):
    # --- Training ---
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)  # (B, 3, 32, 32), (B,)
        
        logits = model(images)  # (B, 10)
        loss = F.cross_entropy(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    scheduler.step()
    avg_loss = epoch_loss / num_batches
    train_losses.append(avg_loss)
    
    # --- Evaluation ---
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)  # (B, 10)
            preds = logits.argmax(dim=1)  # (B,)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    acc = 100.0 * correct / total
    test_accs.append(acc)
    print(f"Epoch {epoch+1:2d}/{num_epochs} | Loss: {avg_loss:.4f} | Test Acc: {acc:.1f}%")

print(f"\nFinal test accuracy: {test_accs[-1]:.1f}%")
if test_accs[-1] >= 70.0:
    print("Target of >70% accuracy achieved!")
else:
    print("Note: More epochs or tuning needed to reach 70%.")

In [ ]:
# ✅ SOLUTION (continued) — Plot training curves

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, num_epochs + 1), train_losses, "b-o", markersize=4)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Training Loss")
ax1.set_title("Training Loss")
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, num_epochs + 1), test_accs, "r-o", markersize=4)
ax2.axhline(y=70, color="gray", linestyle="--", label="70% target")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Test Accuracy (%)")
ax2.set_title("Test Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("CIFAR-10 with Diffusion-Style Building Blocks", fontsize=12)
plt.tight_layout()
plt.show()
print("If accuracy is above 70%, your diffusion-style encoder works!")

---
## Summary

You've now built every convolutional building block you'll need for diffusion models:

| Component | What you learned | Where it shows up in diffusion |
|-----------|-----------------|-------------------------------|
| **Convolution** | Sliding kernel, weight sharing, translation equivariance | Every layer in the U-Net |
| **Padding/Stride/Dilation** | Output size formula; stride-2 for downsampling | Encoder downsampling |
| **Transposed Conv** | Upsampling via zero-insertion; interpolate+conv preferred | Decoder upsampling |
| **Depthwise Separable** | ~9x fewer params for 3x3 convolutions | Some efficient U-Net variants |
| **GroupNorm** | Batch-independent normalization (32 groups standard) | Every block in the U-Net |
| **Residual Connections** | y = F(x) + x; gradient highway; 1x1 projection for dim change | Every block in the U-Net |

In Module 3, we'll add the other key ingredient — **attention mechanisms** — that let the network capture long-range dependencies beyond the local receptive field of convolutions.